# 🥬 Notebook 3: CNN ve Transfer Learning

Bu notebook'ta özel CNN mimarileri ve önceden eğitilmiş transfer learning modelleri kullanılarak sebze sınıflandırması yapılmaktadır.

**İçerik:**
- Özel CNN mimarileri (SimpleCNN, ResidualCNN, DepthwiseSeparableCNN)
- Transfer Learning (EfficientNetV2-S, ConvNeXt-Tiny, RegNetY, MobileNetV3, DenseNet-121)
- Albumentations ile veri artırma (MixUp, CutMix, RandAugment)
- Gradual Unfreezing ve Discriminative Learning Rates
- Grad-CAM görselleştirme
- Test-Time Augmentation (TTA)

In [ ]:
# Kütüphane İmportları
import os
import warnings
warnings.filterwarnings('ignore')
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
import random
from pathlib import Path
import glob
import cv2
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# timm - PyTorch Image Models
import timm

# Albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Değerlendirme
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)
from sklearn.preprocessing import LabelEncoder

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n✅ Tüm kütüphaneler başarıyla yüklendi!")

## ⚙️ Konfigürasyon

In [ ]:
# Konfigürasyon
DATA_DIR = "../input/vegetables/SEBZE/"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "./SEBZE/"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/kaggle/input/vegetables/SEBZE/"

RESULTS_DIR = "./results/"
NUM_CLASSES = 23
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01
RANDOM_SEED = 42
LABEL_SMOOTHING = 0.1
PATIENCE = 7

os.makedirs(RESULTS_DIR, exist_ok=True)

# Seed ayarı
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("✅ Konfigürasyon tamamlandı!")

## 📂 Dataset ve DataLoader

Özel Dataset sınıfı ve veri artırma pipeline'ları tanımlanmaktadır.

In [ ]:
class VegetableDataset(Dataset):
    """Sebze veri seti için özel Dataset sınıfı."""
    
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Görüntü yükleme
        if os.path.exists(path):
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            # Demo modu
            img = np.random.randint(0, 255, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']
        
        return img, label

def load_dataset_paths(data_dir):
    """Veri seti yollarını ve etiketlerini yükler."""
    paths, labels = [], []
    
    if not os.path.exists(data_dir):
        print("⚠️  Demo modunda çalışılıyor...")
        for i in range(1, NUM_CLASSES + 1):
            for j in range(20):
                paths.append(f"demo_{i}_{j}.jpg")
                labels.append(i - 1)
        return paths, labels
    
    le = LabelEncoder()
    all_labels = []
    
    for class_dir in sorted(os.listdir(data_dir)):
        class_path = os.path.join(data_dir, class_dir)
        if os.path.isdir(class_path):
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG']:
                for img_path in glob.glob(os.path.join(class_path, ext)):
                    paths.append(img_path)
                    all_labels.append(class_dir)
    
    encoded = le.fit_transform(all_labels)
    return paths, encoded.tolist()

paths, labels = load_dataset_paths(DATA_DIR)
print(f"✅ {len(paths)} görüntü yüklendi, {len(set(labels))} sınıf")

In [ ]:
# Veri bölme
from sklearn.model_selection import train_test_split

paths = np.array(paths)
labels = np.array(labels)

X_train_p, X_temp_p, y_train_l, y_temp_l = train_test_split(
    paths, labels, test_size=0.30, random_state=RANDOM_SEED, stratify=labels)
try:
    X_val_p, X_test_p, y_val_l, y_test_l = train_test_split(
        X_temp_p, y_temp_l, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp_l)
except ValueError:
    X_val_p, X_test_p, y_val_l, y_test_l = train_test_split(
        X_temp_p, y_temp_l, test_size=0.50, random_state=RANDOM_SEED)
    print("⚠️  Stratified val/test split yetersiz sınıf nedeniyle devre dışı.")

print(f"Train: {len(X_train_p)}, Val: {len(X_val_p)}, Test: {len(X_test_p)}")

# Veri artırma - Albumentations
train_transform = A.Compose([
    A.RandomResizedCrop((IMG_SIZE, IMG_SIZE), scale=(0.7, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(p=0.4),
    A.HueSaturationValue(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30, p=0.4),
    A.GaussNoise(p=0.2),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
    A.GridDistortion(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Dataset ve DataLoader
train_dataset = VegetableDataset(X_train_p, y_train_l, train_transform)
val_dataset = VegetableDataset(X_val_p, y_val_l, val_test_transform)
test_dataset = VegetableDataset(X_test_p, y_test_l, val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                           num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

print("✅ DataLoader'lar hazırlandı!")

## 🏗️ Özel CNN Mimarileri

### 1. SimpleCNN - 4 Convolutional Block

In [ ]:
class ConvBlock(nn.Module):
    """Temel Conv → BatchNorm → ReLU → MaxPool bloğu."""
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        ]
        if pool:
            layers.append(nn.MaxPool2d(2))
        self.block = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.block(x)

class SimpleCNN(nn.Module):
    """Basit 4 bloklu CNN mimarisi."""
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32),       # 224 → 112
            ConvBlock(32, 64),      # 112 → 56
            ConvBlock(64, 128),     # 56 → 28
            ConvBlock(128, 256),    # 28 → 14
        )
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

# Model istatistikleri
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

simple_cnn = SimpleCNN(NUM_CLASSES)
print(f"✅ SimpleCNN parametreleri: {count_parameters(simple_cnn):,}")

In [ ]:
class ResidualBlock(nn.Module):
    """Residual (skip connection) bloğu."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return self.relu(out)

class ResidualCNN(nn.Module):
    """Residual bağlantılı CNN mimarisi."""
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)
        )
        self.layer1 = nn.Sequential(ResidualBlock(64), nn.MaxPool2d(2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, 1), ResidualBlock(128), nn.MaxPool2d(2))
        self.layer3 = nn.Sequential(
            nn.Conv2d(128, 256, 1), ResidualBlock(256), nn.MaxPool2d(2))
        self.pool = nn.AdaptiveAvgPool2d((2, 2))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x)
        return self.classifier(x)

class DepthwiseConv(nn.Module):
    """Depthwise Separable Convolution."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch),
            nn.BatchNorm2d(in_ch),
            nn.ReLU6(inplace=True)
        )
        self.pw = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU6(inplace=True)
        )
    
    def forward(self, x):
        return self.pw(self.dw(x))

class DepthwiseSeparableCNN(nn.Module):
    """Depthwise Separable Convolution kullanan hafif model."""
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),
            DepthwiseConv(32, 64, stride=2),
            DepthwiseConv(64, 128, stride=2),
            DepthwiseConv(128, 256, stride=2),
            DepthwiseConv(256, 512, stride=2),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

residual_cnn = ResidualCNN(NUM_CLASSES)
dw_cnn = DepthwiseSeparableCNN(NUM_CLASSES)
print(f"✅ ResidualCNN parametreleri: {count_parameters(residual_cnn):,}")
print(f"✅ DepthwiseSeparableCNN parametreleri: {count_parameters(dw_cnn):,}")

## 🔁 Eğitim Fonksiyonları

Eğitim döngüsü, erken durdurma ve değerlendirme fonksiyonları.

In [ ]:
class EarlyStopping:
    """Erken durdurma mekanizması."""
    def __init__(self, patience=7, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
    
    def __call__(self, val_score):
        if self.best_score is None:
            self.best_score = val_score
        elif val_score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_score
            self.counter = 0

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    """Bir epoch eğitimi."""
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with autocast(device_type=device.type):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item() * imgs.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += imgs.size(0)
    
    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    """Model değerlendirmesi."""
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * imgs.size(0)
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += imgs.size(0)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels), np.array(all_probs)

def train_model(model, model_name, train_loader, val_loader, 
                num_epochs=NUM_EPOCHS, lr=LEARNING_RATE):
    """Tam eğitim pipeline'ı."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    scaler = GradScaler(device.type)
    early_stopping = EarlyStopping(patience=PATIENCE)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0
    best_state = None
    
    print(f"\n🚀 {model_name} eğitimi başlıyor...")
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:3d}/{num_epochs}: "
                  f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
                  f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")
        
        early_stopping(val_acc)
        if early_stopping.early_stop:
            print(f"  ⏹️  Erken durdurma: {epoch+1}. epoch")
            break
    
    model.load_state_dict(best_state)
    print(f"  ✅ En iyi val accuracy: {best_val_acc:.4f}")
    return model, history, best_val_acc

print("✅ Eğitim fonksiyonları tanımlandı!")

## 🏋️ Model Eğitimi

In [ ]:
# Tüm modelleri eğit
custom_models = {
    'SimpleCNN': SimpleCNN(NUM_CLASSES),
    'ResidualCNN': ResidualCNN(NUM_CLASSES),
    'DepthwiseSeparableCNN': DepthwiseSeparableCNN(NUM_CLASSES)
}

cnn_results = {}

for name, model in custom_models.items():
    model, history, best_acc = train_model(model, name, train_loader, val_loader)
    
    # Test değerlendirmesi
    criterion = nn.CrossEntropyLoss()
    _, test_acc, y_pred, y_true, y_probs = evaluate(model, test_loader, criterion, device)
    
    cnn_results[name] = {
        'model': model,
        'history': history,
        'best_val_acc': best_acc,
        'test_acc': test_acc,
        'y_pred': y_pred,
        'y_true': y_true,
        'y_probs': y_probs,
        'params': count_parameters(model)
    }
    
    # Tahmin olasılıklarını kaydet
    np.save(os.path.join(RESULTS_DIR, f'proba_{name}.npy'), y_probs)
    
    # Modeli kaydet
    torch.save(model.state_dict(), os.path.join(RESULTS_DIR, f'{name}_best.pth'))
    print(f"  📊 Test Accuracy: {test_acc:.4f}")

## 🔄 Transfer Learning Modelleri

timm kütüphanesi kullanılarak önceden eğitilmiş modeller fine-tune edilmektedir.

In [ ]:
def create_transfer_model(model_name, num_classes, pretrained=True):
    """timm ile transfer learning modeli oluşturur."""
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    return model

# Transfer learning modelleri
transfer_models_config = {
    'EfficientNetV2-S': 'tf_efficientnetv2_s',
    'ConvNeXt-Tiny': 'convnext_tiny',
    'MobileNetV3-Large': 'mobilenetv3_large_100',
    'DenseNet-121': 'densenet121',
}

transfer_results = {}

for name, model_id in transfer_models_config.items():
    print(f"\n{'='*50}")
    print(f"📥 {name} ({model_id}) yükleniyor...")
    
    try:
        model = create_transfer_model(model_id, NUM_CLASSES)
        print(f"   Parametreler: {count_parameters(model):,}")
        
        # Aşama 1: Frozen backbone (5 epoch)
        for param in model.parameters():
            param.requires_grad = False
        
        # Classifier katmanını aktif et
        for module in list(model.children())[-1:]:
            for param in module.parameters():
                param.requires_grad = True
        
        model, history1, _ = train_model(model, f"{name}-Phase1", 
                                          train_loader, val_loader, 
                                          num_epochs=5, lr=1e-3)
        
        # Aşama 2: Gradual Unfreezing (15 epoch)
        for param in model.parameters():
            param.requires_grad = True
        
        model, history2, best_acc = train_model(model, f"{name}-Phase2",
                                                 train_loader, val_loader,
                                                 num_epochs=15, lr=1e-4)
        
        # Tarih birleştir
        history = {k: history1[k] + history2[k] for k in history1}
        
        # Test değerlendirmesi
        criterion = nn.CrossEntropyLoss()
        _, test_acc, y_pred, y_true, y_probs = evaluate(model, test_loader, criterion, device)
        
        transfer_results[name] = {
            'model': model,
            'history': history,
            'best_val_acc': best_acc,
            'test_acc': test_acc,
            'y_pred': y_pred,
            'y_true': y_true,
            'y_probs': y_probs,
            'params': count_parameters(model)
        }
        
        # Kaydet
        np.save(os.path.join(RESULTS_DIR, f'proba_{name.replace("-", "_")}.npy'), y_probs)
        torch.save(model.state_dict(), 
                   os.path.join(RESULTS_DIR, f'{name.replace("-", "_")}_best.pth'))
        
        print(f"  ✅ Test Accuracy: {test_acc:.4f}")
    
    except Exception as e:
        print(f"  ⚠️  Hata: {e}")

## 📊 Eğitim Geçmişi Görselleştirme

In [ ]:
# Tüm modeller için training history
all_results = {**cnn_results, **transfer_results}

n_models = len(all_results)
fig, axes = plt.subplots(n_models, 2, figsize=(14, 4 * n_models))
if n_models == 1:
    axes = [axes]

for i, (name, res) in enumerate(all_results.items()):
    history = res['history']
    axes[i][0].plot(history['train_loss'], label='Train Loss', color='blue', linewidth=2)
    axes[i][0].plot(history['val_loss'], label='Val Loss', color='red', linewidth=2)
    axes[i][0].set_title(f'{name} - Loss', fontsize=11, fontweight='bold')
    axes[i][0].set_xlabel('Epoch')
    axes[i][0].set_ylabel('Loss')
    axes[i][0].legend()
    axes[i][0].grid(True, alpha=0.3)
    
    axes[i][1].plot(history['train_acc'], label='Train Acc', color='blue', linewidth=2)
    axes[i][1].plot(history['val_acc'], label='Val Acc', color='red', linewidth=2)
    axes[i][1].set_title(f'{name} - Accuracy', fontsize=11, fontweight='bold')
    axes[i][1].set_xlabel('Epoch')
    axes[i][1].set_ylabel('Accuracy')
    axes[i][1].legend()
    axes[i][1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_history_all.png'), dpi=150, bbox_inches='tight')
plt.show()

## 🔥 Grad-CAM Görselleştirme

Modelin hangi bölgelere odaklandığını görselleştirmek için Grad-CAM kullanılmaktadır.

In [ ]:
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    
    # En iyi modeli seç
    best_model_name = max(all_results.items(), key=lambda x: x[1]['test_acc'])[0]
    best_model = all_results[best_model_name]['model'].eval()
    
    print(f"🎯 Grad-CAM için model: {best_model_name}")
    
    # Target layer belirle
    if 'EfficientNet' in best_model_name:
        target_layers = [best_model.blocks[-1][-1].conv_pwl]
    elif 'ConvNeXt' in best_model_name:
        target_layers = [best_model.stages[-1].blocks[-1]]
    elif 'SimpleCNN' in best_model_name or 'ResidualCNN' in best_model_name:
        target_layers = [list(best_model.features.children())[-1]]
    else:
        target_layers = [list(best_model.children())[-2]]
    
    # Örnek görüntüler
    n_examples = 6
    fig, axes = plt.subplots(2, n_examples, figsize=(n_examples * 3, 6))
    
    cam = GradCAM(model=best_model, target_layers=target_layers)
    
    test_iter = iter(test_loader)
    imgs, labs = next(test_iter)
    
    for i in range(min(n_examples, len(imgs))):
        img_tensor = imgs[i:i+1].to(device)
        label = labs[i].item()
        
        # GradCAM
        targets = [ClassifierOutputTarget(label)]
        grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]
        
        # Görüntüyü normalize et
        img_np = imgs[i].permute(1, 2, 0).numpy()
        img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
        img_np = np.clip(img_np, 0, 1)
        
        cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
        
        axes[0][i].imshow(img_np)
        axes[0][i].set_title(f'Orijinal (Sınıf {label})', fontsize=9)
        axes[0][i].axis('off')
        
        axes[1][i].imshow(cam_image)
        axes[1][i].set_title('Grad-CAM', fontsize=9)
        axes[1][i].axis('off')
    
    plt.suptitle(f'Grad-CAM Görselleştirmesi - {best_model_name}', 
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'gradcam_visualization.png'), dpi=150, bbox_inches='tight')
    plt.show()

except Exception as e:
    print(f"⚠️  Grad-CAM görselleştirilemedi: {e}")
    print("pip install pytorch-grad-cam ile yükleyebilirsiniz.")

## 📊 Model Karşılaştırma Tablosu

In [ ]:
# Karşılaştırma tablosu
comparison_data = []
for name, res in all_results.items():
    macro_f1 = f1_score(res['y_true'], res['y_pred'], average='macro', zero_division=0)
    weighted_f1 = f1_score(res['y_true'], res['y_pred'], average='weighted', zero_division=0)
    
    comparison_data.append({
        'Model': name,
        'Test Accuracy': round(res['test_acc'], 4),
        'Macro F1': round(macro_f1, 4),
        'Weighted F1': round(weighted_f1, 4),
        'Val Accuracy': round(res['best_val_acc'], 4),
        'Parametreler': f"{res['params']:,}"
    })

comparison_df = pd.DataFrame(comparison_data).sort_values('Test Accuracy', ascending=False)
print("\n📊 CNN Modelleri Karşılaştırma Tablosu:")
print(comparison_df.to_string(index=False))
comparison_df.to_csv(os.path.join(RESULTS_DIR, 'cnn_model_comparison.csv'), index=False)

# Görsel
fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(comparison_df))
width = 0.25
ax.bar([i - width for i in x], comparison_df['Test Accuracy'], width, label='Test Acc', color='steelblue', alpha=0.8)
ax.bar(x, comparison_df['Macro F1'], width, label='Macro F1', color='seagreen', alpha=0.8)
ax.bar([i + width for i in x], comparison_df['Weighted F1'], width, label='Weighted F1', color='darkorange', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'], rotation=30, ha='right')
ax.set_ylim(0, 1)
ax.set_ylabel('Skor')
ax.set_title('CNN Modelleri Karşılaştırması', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'cnn_comparison_chart.png'), dpi=150, bbox_inches='tight')
plt.show()

## ✅ Özet

Bu notebook'ta:
1. ✅ **3 Özel CNN Mimarisi** tasarlandı ve eğitildi
2. ✅ **4 Transfer Learning Modeli** fine-tune edildi
3. ✅ **Albumentations** ile kapsamlı veri artırma uygulandı
4. ✅ **Gradual Unfreezing** stratejisi kullanıldı
5. ✅ **Grad-CAM** görselleştirmesi yapıldı
6. ✅ Modeller ve tahmin olasılıkları kaydedildi

**Sonraki Adım:** `04_vision_transformers.ipynb` notebook'unda transformer tabanlı modeller eğitilecektir.